# NHL Trap Games

Definition: A trap game is about a good team underperforming when they're expected to win

In [1]:
import sys
import os
import polars as pl
from tqdm import tqdm

base_dir = '../'
sys.path.insert(1, base_dir)

import src.config as cfg
import src.utils as utils
import src.sportsipy_utils as sportsipy_utils

## Pull Schedule for Each NHL Team

Note: We are using *5 v 5* results to focus on a team's true underlying ability

In [2]:
# Warning: NHL Schedules not loading - https://github.com/davidjkrause/sportsipy/issues/14
# sportsipy_utils.pull_nhl_schedule('TOR', 2024)

In [57]:
# https://www.naturalstattrick.com/games.php?fromseason=20242025&thruseason=20242025&stype=2&sit=5v5&loc=B&team=All&rate=n
schedule_folder = f'{base_dir}data/natural_stat_trick'

df_games = []
for filename in tqdm(os.listdir(schedule_folder)):
    if 'games_' in filename:
        schedule_filepath = os.path.join(schedule_folder, filename)
        df_games_filepath = pl.scan_csv(schedule_filepath, null_values=["-"]).collect()
        if not df_games_filepath.is_empty():
            df_games += [df_games_filepath]
    
df_games = (pl.concat(df_games)
            .unique(subset=['Game', 'Team'], keep='first')
            .with_columns(
                pl.col('Game').str.split(" - ").list.get(0).str.strip_chars().str.strptime(pl.Datetime,'%Y-%m-%d').alias('Date')
            )
            .select(['Game', 'Team', 'Date', 'GF', 'GA', 'GF%', 'xGF', 'xGA', 'xGF%', 'SH%', 'SV%' ,'PDO'])
           )
utils.logger.info(f"Loaded {df_games.shape[0]/2} unique games from natural stattrick") 
df_games.head(5)

100%|██████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 92.41it/s]
2025-09-13 11:34:07,815 [1402038357.py:19] [INFO] Loaded 21528.0 unique games from natural stattrick


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64
""" 2021-01-24 - Avalanche 1, Duc…","""Anaheim Ducks""",2021-01-24 00:00:00,2,1,66.67,1.39,2.05,40.42,18.18,96.15,1.143
""" 2016-12-29 - Rangers 6, Coyot…","""New York Rangers""",2016-12-29 00:00:00,1,2,33.33,1.79,1.34,57.29,5.26,86.67,0.919
""" 2010-12-31 - Senators 3, Blue…","""Ottawa Senators""",2010-12-31 00:00:00,2,3,40.0,1.72,1.33,56.46,7.14,76.92,0.841
""" 2017-02-18 - Blues 2, Sabres …","""Buffalo Sabres""",2017-02-18 00:00:00,2,0,100.0,0.74,1.8,28.98,12.5,100.0,1.125
""" 2019-03-11 - Senators 2, Flye…","""Ottawa Senators""",2019-03-11 00:00:00,2,3,40.0,1.35,3.02,30.82,10.0,89.66,0.997


In [82]:
# Warning: for 1000 pairs of (game, team), there were no 5v5 goals scored extracted GF/GA numbers 
df_missing = df_games.filter(pl.col("GF%").is_null())
utils.logger.info(f"Found {df_missing.shape[0]} / {df_games.shape[0]} rows without any 5v5 goals")

2025-09-13 11:49:00,934 [3340401753.py:3] [INFO] Found 1050 / 43056 rows without any 5v5 goals


## Identify Trap Games

In [127]:
strength_col = "5v5 Points"
strong_threshold = 1.1
weak_threshold = 0.9

df_games_processed = (
    df_games.filter(pl.col("GF%").is_not_null())
      .unique(subset=['Game', 'Team'], keep='first')
      .sort("Game", descending=False)
      .with_columns([
        pl.when(pl.col("Date").dt.month() >= 7).then(pl.col("Date").dt.year() + 1)
            .otherwise(pl.col("Date").dt.year())
            .alias("Season"),
          pl.when(pl.col("GF%")>50).then(pl.lit(2))
              .when(pl.col("GF%")<50).then(pl.lit(0))
              .otherwise(pl.lit(1))
          .alias("5v5 Points")
      ])
    .with_columns([
        pl.col(strength_col).cum_sum().over(["Team", "Season"]).alias("Season to Date Total incl Current"),
        pl.col("GF%").cum_count().over(["Team", "Season"]).alias("# Games Season to Date incl Current")
    ])
    .with_columns(
        ((pl.col("Season to Date Total incl Current") - pl.col(strength_col))/
         (pl.col("# Games Season to Date incl Current") - 1)).alias(f"Season to Date Average {strength_col}")
    )
    .with_columns([
        ((pl.col(f"Season to Date Average {strength_col}") > strong_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_strong"),
        ((pl.col(f"Season to Date Average {strength_col}") < weak_threshold) 
         & (pl.col("# Games Season to Date incl Current") > 20)).alias("is_weak")
    ])
)  
                     
display(df_games_processed.tail(5))
df_games_processed.mean()

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool
""" 2025-04-17 - Islanders 1, Blu…","""Columbus Blue Jackets""",2025-04-17 00:00:00,6,1,85.71,2.9,2.92,49.86,23.08,97.14,1.202,2025,2,78,78,0.987013,false,false
""" 2025-04-17 - Lightning 0, Ran…","""New York Rangers""",2025-04-17 00:00:00,3,0,100.0,1.89,2.17,46.51,15.0,100.0,1.15,2025,2,82,81,1.0,false,false
""" 2025-04-17 - Lightning 0, Ran…","""Tampa Bay Lightning""",2025-04-17 00:00:00,0,3,0.0,2.17,1.89,53.49,0.0,85.0,0.85,2025,0,94,81,1.175,true,false
""" 2025-04-17 - Red Wings 3, Map…","""Detroit Red Wings""",2025-04-17 00:00:00,2,2,50.0,2.25,1.93,53.84,6.9,87.5,0.944,2025,1,78,79,0.987179,false,false
""" 2025-04-17 - Red Wings 3, Map…","""Toronto Maple Leafs""",2025-04-17 00:00:00,2,2,50.0,1.93,2.25,46.16,12.5,93.1,1.056,2025,1,88,77,1.144737,true,false


Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak
str,str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,2016-09-02 12:12:30.749892,1.927296,1.927272,50.000793,1.863623,1.863614,50.000135,8.501321,91.498861,1.000008,2016.636957,1.0,39.156192,39.141266,NaN,0.200495,0.200519


In [128]:
# Double check season derivation is reasonable given lockout in the 2012-2013 season and pandemic in 2019-2020/2020-2021 seasons
df_games_processed.group_by("Season").agg(pl.col("Game").count()).sort("Game").head(5)

Season,Game
i32,u32
2013,1404
2021,1698
2020,2126
2008,2352
2009,2358


In [158]:
df_opponent = df_games_processed.select([
    pl.col("Game"),
    pl.col("Team").alias("Opponent"),
    pl.col(f"Season to Date Average {strength_col}").alias(f"Opponent Season to Date Average {strength_col}"),
    pl.col("is_strong").alias("Opponent is_strong"),
    pl.col("is_weak").alias("Opponent is_weak")
])

df_games_with_opponent = (
    df_games_processed.join(df_opponent, on="Game", how="left")
    .filter(pl.col("Team") != pl.col("Opponent"))
    .with_columns(
        (pl.col(f"Season to Date Average {strength_col}") - pl.col(f"Opponent Season to Date Average {strength_col}")).alias(f"Delta {strength_col}")
    )

    # Tag previous opponent
    .sort(["Team", "Date"])
    .with_columns([
        pl.col(col).shift(1).over(["Team", "Season"]).alias(f"Previous {col}")
        for col in ["Opponent", "Opponent is_strong", "Opponent is_weak"]
    ])

    # Tag Trap Games
    .with_columns(
        pl.when((pl.col("is_strong"))&(pl.col("Opponent is_weak"))&(pl.col("Previous Opponent is_strong"))).then(pl.lit(True))
        .otherwise(pl.lit(False)).alias("Trap Game")
    )
)

assert df_games_with_opponent.shape[0] == df_games_processed.shape[0]

df_games_with_opponent.filter(pl.col("Trap Game"))

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak,Opponent,Opponent Season to Date Average 5v5 Points,Opponent is_strong,Opponent is_weak,Delta 5v5 Points,Previous Opponent,Previous Opponent is_strong,Previous Opponent is_weak,Trap Game
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str,f64,bool,bool,f64,str,bool,bool,bool
""" 2007-11-30 - Ducks 1, Oilers …","""Anaheim Ducks""",2007-11-30 00:00:00,1,4,20.0,0.59,2.12,21.9,7.14,84.0,0.911,2008,0,27,25,1.125,true,false,"""Edmonton Oilers""",0.583333,false,true,0.541667,"""Calgary Flames""",true,false,true
""" 2008-12-10 - Blues 2, Ducks 4""","""Anaheim Ducks""",2008-12-10 00:00:00,2,1,66.67,2.31,1.5,60.58,8.33,94.12,1.025,2009,2,34,28,1.185185,true,false,"""St Louis Blues""",0.791667,false,true,0.393519,"""Columbus Blue Jackets""",true,false,true
""" 2008-12-14 - Wild 2, Ducks 4""","""Anaheim Ducks""",2008-12-14 00:00:00,4,1,80.0,1.71,1.7,50.05,21.05,94.44,1.155,2009,2,36,30,1.172414,true,false,"""Minnesota Wild""",0.84,false,true,0.332414,"""San Jose Sharks""",true,false,true
""" 2010-03-19 - Islanders 4, Duc…","""Anaheim Ducks""",2010-03-19 00:00:00,2,2,50.0,2.69,2.18,55.27,6.06,92.86,0.989,2010,1,73,66,1.107692,true,false,"""New York Islanders""",0.823529,false,true,0.284163,"""Chicago Blackhawks""",true,false,true
""" 2013-03-18 - Sharks 3, Ducks …","""Anaheim Ducks""",2013-03-18 00:00:00,3,3,50.0,1.59,2.22,41.72,13.04,90.32,1.034,2013,1,36,28,1.296296,true,false,"""San Jose Sharks""",0.875,false,true,0.421296,"""St Louis Blues""",true,false,true
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
""" 2024-12-03 - Blues 4, Jets 1""","""Winnipeg Jets""",2024-12-03 00:00:00,0,1,0.0,1.54,2.36,39.54,0.0,95.0,0.95,2025,0,28,26,1.12,true,false,"""St Louis Blues""",0.875,false,true,0.245,"""Dallas Stars""",true,false,true
""" 2024-12-14 - Canadiens 2, Jet…","""Winnipeg Jets""",2024-12-14 00:00:00,1,2,33.33,2.02,1.77,53.29,4.76,89.47,0.942,2025,0,35,32,1.129032,true,false,"""Montreal Canadiens""",0.62963,false,true,0.499403,"""Vegas Golden Knights""",true,false,true
""" 2024-12-28 - Senators 2, Jets…","""Winnipeg Jets""",2024-12-28 00:00:00,3,0,100.0,2.39,1.72,58.11,16.67,100.0,1.167,2025,2,42,37,1.111111,true,false,"""Ottawa Senators""",0.852941,false,true,0.25817,"""Toronto Maple Leafs""",true,false,true


In [161]:
df_leafs_sample = df_games_with_opponent.filter((pl.col("Team")=="Toronto Maple Leafs")&(pl.col("Season")==2025))
display(df_leafs_sample.head(5))

df_leafs_sample.write_csv(f"{base_dir}data/derived/Leafs Trap Games Sample.csv")

Game,Team,Date,GF,GA,GF%,xGF,xGA,xGF%,SH%,SV%,PDO,Season,5v5 Points,Season to Date Total incl Current,# Games Season to Date incl Current,Season to Date Average 5v5 Points,is_strong,is_weak,Opponent,Opponent Season to Date Average 5v5 Points,Opponent is_strong,Opponent is_weak,Delta 5v5 Points,Previous Opponent,Previous Opponent is_strong,Previous Opponent is_weak,Trap Game
str,str,datetime[μs],i64,i64,f64,f64,f64,f64,f64,f64,f64,i32,i32,i32,u32,f64,bool,bool,str,f64,bool,bool,f64,str,bool,bool,bool
""" 2024-10-10 - Maple Leafs 4, D…","""Toronto Maple Leafs""",2024-10-10 00:00:00,4,1,80.0,1.66,1.45,53.49,20.0,92.86,1.129,2025,2,2,1,NaN,false,false,"""New Jersey Devils""",2.0,false,false,NaN,null,null,null,false
""" 2024-10-12 - Penguins 2, Mapl…","""Toronto Maple Leafs""",2024-10-12 00:00:00,3,1,75.0,1.45,1.03,58.57,12.0,93.75,1.058,2025,2,4,2,2.0,false,false,"""Pittsburgh Penguins""",1.0,false,false,1.0,"""New Jersey Devils""",false,false,false
""" 2024-10-16 - Kings 2, Maple L…","""Toronto Maple Leafs""",2024-10-16 00:00:00,4,2,66.67,0.94,4.14,18.56,23.53,93.33,1.169,2025,2,6,3,2.0,false,false,"""Los Angeles Kings""",2.0,false,false,0.0,"""Pittsburgh Penguins""",false,false,false
""" 2024-10-19 - Rangers 4, Maple…","""Toronto Maple Leafs""",2024-10-19 00:00:00,1,1,50.0,2.96,2.39,55.27,3.7,95.65,0.994,2025,1,7,4,2.0,false,false,"""New York Rangers""",2.0,false,false,0.0,"""Los Angeles Kings""",false,false,false
""" 2024-10-21 - Lightning 2, Map…","""Toronto Maple Leafs""",2024-10-21 00:00:00,4,1,80.0,2.13,2.28,48.3,19.05,95.0,1.14,2025,2,9,5,1.75,false,false,"""Tampa Bay Lightning""",1.5,false,false,0.25,"""New York Rangers""",false,false,false
